# Day 3 Tutorial：Ridge 与 L2 正则化

## Goal

从单特征线性预测出发，只改变 `alpha`，观察权重压缩与 train/validation RMSE。

## Setup

固定同尺度人工数据；这不是 ESOL 或下游任务结果。

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

X_train = np.array([[0.0], [1.0], [2.0], [3.0], [4.0], [5.0]])
y_train = np.array([0.2, 1.1, 1.9, 3.2, 3.9, 5.1])
X_valid = np.array([[1.5], [3.5], [5.5]])
y_valid = np.array([1.4, 3.6, 5.4])

assert X_train.shape == (6, 1)
assert y_train.shape == (6,)
print('data shapes:', X_train.shape, y_train.shape, X_valid.shape, y_valid.shape)

data shapes: (6, 1) (6,) (3, 1) (3,)


## Steps

先训练固定模型并复算一个预测，再建立 `alpha` 对照表。

In [2]:
fixed_model = Ridge(alpha=1.0)
fixed_model.fit(X_train, y_train)
fixed_prediction = fixed_model.predict(X_valid)

manual_first = (
    fixed_model.intercept_
    + fixed_model.coef_[0] * X_valid[0, 0]
)
assert np.isclose(manual_first, fixed_prediction[0])
print('weight:', round(float(fixed_model.coef_[0]), 4))
print('intercept:', round(float(fixed_model.intercept_), 4))
print('first prediction:', round(float(manual_first), 4))

weight: 0.9243
intercept: 0.2559
first prediction: 1.6423


In [3]:
def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))

alpha_values = [0.01, 0.1, 1.0, 10.0, 100.0]
records = []

for alpha in alpha_values:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    records.append({
        'alpha': alpha,
        'coefficient': float(model.coef_[0]),
        'absolute_coefficient': float(abs(model.coef_[0])),
        'train_rmse': rmse(y_train, model.predict(X_train)),
        'valid_rmse': rmse(y_valid, model.predict(X_valid)),
    })

results = pd.DataFrame(records)
results

,alpha,coefficient,absolute_coefficient,train_rmse,valid_rmse
0,0.01,0.976585,0.976585,0.118460,0.127343
1,0.10,0.971591,0.971591,0.118835,0.127147
2,1.00,0.924324,0.924324,0.148891,0.157328
3,10.00,0.621818,0.621818,0.618286,0.683860
4,100.00,0.145532,0.145532,1.425177,1.589052


## Checks

核对候选完整、数值有限，并检查极大 `alpha` 的权重更小。

In [4]:
assert results['alpha'].tolist() == alpha_values
assert np.isfinite(results.to_numpy()).all()
small_alpha_weight = results.loc[
    results['alpha'] == 0.01, 'absolute_coefficient'
].iloc[0]
large_alpha_weight = results.loc[
    results['alpha'] == 100.0, 'absolute_coefficient'
].iloc[0]
assert large_alpha_weight < small_alpha_weight
print('Ridge behavior checks passed')

Ridge behavior checks passed


## Next Steps

独立完成 `03_exercises.md`。个人练习需在 `learning_outputs/day03_ridge/` 重新运行并保存自己生成的 `alpha_sensitivity.csv`。